In [26]:
print("📦 Installing Kafka and dependencies...")
print("⏳ This may take 2-3 minutes...\n")

# Install Java
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y openjdk-11-jdk -qq > /dev/null 2>&1
print("✅ Java installed")

# Download and extract Kafka
import os

if not os.path.exists('kafka_2.13-3.6.1.tgz'):
    print("⬇️  Downloading Kafka...")
    !wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
else:
    print("✅ Kafka archive already exists")

print("📦 Extracting Kafka...")
!tar -xzf kafka_2.13-3.6.1.tgz
print("✅ Kafka extracted")

# Install Python library
!pip install kafka-python -q
print("✅ kafka-python installed")

# Verify installation
if os.path.exists('kafka_2.13-3.6.1/bin/zookeeper-server-start.sh'):
    print("✅ Kafka files verified")
else:
    print("❌ Kafka installation failed - please re-run this cell")

print("\n✅ Installation complete!\n")

📦 Installing Kafka and dependencies...
⏳ This may take 2-3 minutes...

✅ Java installed
✅ Kafka archive already exists
📦 Extracting Kafka...
✅ Kafka extracted
✅ kafka-python installed
✅ Kafka files verified

✅ Installation complete!



In [27]:
import subprocess
import time
import os

print("🚀 Starting Zookeeper and Kafka...\n")

KAFKA_HOME = 'kafka_2.13-3.6.1'

if not os.path.exists(KAFKA_HOME):
    print("❌ Kafka directory not found! Please re-run Cell 1")
else:
    # Start Zookeeper
    zookeeper = subprocess.Popen(
        [f'{KAFKA_HOME}/bin/zookeeper-server-start.sh', f'{KAFKA_HOME}/config/zookeeper.properties'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅ Zookeeper started")
    time.sleep(10)  # Longer wait

    # Start Kafka broker
    kafka_server = subprocess.Popen(
        [f'{KAFKA_HOME}/bin/kafka-server-start.sh', f'{KAFKA_HOME}/config/server.properties'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("✅ Kafka server started")
    time.sleep(20)  # Longer wait for Kafka

    print("\n🎉 Kafka is ready!")
    print("⏰ Wait 10 more seconds, then run the next cells\n")

🚀 Starting Zookeeper and Kafka...

✅ Zookeeper started
✅ Kafka server started

🎉 Kafka is ready!
⏰ Wait 10 more seconds, then run the next cells



In [28]:
from kafka import KafkaProducer, KafkaConsumer
import json
import time
import threading

KAFKA_BROKER = 'localhost:9092'

def serialize(data):
    """Convert Python dict to JSON bytes for Kafka"""
    return json.dumps(data).encode('utf-8')

def deserialize(data):
    """Convert JSON bytes from Kafka to Python dict"""
    return json.loads(data.decode('utf-8'))

print("✅ Setup Complete")

✅ Setup Complete


In [29]:
class SensorAgent:


    def __init__(self):
        self.agent_id = "SensorAgent"
        try:
            self.producer = KafkaProducer(
                bootstrap_servers=KAFKA_BROKER,
                value_serializer=serialize,
                max_block_ms=5000
            )
            self.consumer = KafkaConsumer(
                'commands',
                bootstrap_servers=KAFKA_BROKER,
                value_deserializer=deserialize,
                auto_offset_reset='earliest',
                group_id='sensor-group',
                consumer_timeout_ms=8000
            )
            print(f"🔵 {self.agent_id} initialized successfully")
        except Exception as e:
            print(f"❌ Error initializing {self.agent_id}: {e}")

    def publish_sensor_data(self):
        """Publish sensor readings to Kafka"""
        readings = [
            {'sensor': 'temperature', 'value': 25.5},
            {'sensor': 'humidity', 'value': 60.0},
            {'sensor': 'pressure', 'value': 1013.2}
        ]

        try:
            for reading in readings:
                self.producer.send('sensor.data', reading)
                self.producer.flush()
                print(f"📤 [{self.agent_id}] Published: {reading}")
                time.sleep(1)
        except Exception as e:
            print(f"❌ Error publishing data: {e}")

    def listen_for_commands(self):
        """Subscribe to commands topic"""
        print(f"👂 [{self.agent_id}] Listening for commands...")
        try:
            for message in self.consumer:
                command = message.value
                print(f"📥 [{self.agent_id}] Received command: {command}")
        except:
            pass

    def run(self):
        """Execute agent: publish then subscribe"""
        self.publish_sensor_data()
        self.listen_for_commands()
        self.producer.close()
        self.consumer.close()
        print(f"✅ [{self.agent_id}] finished\n")

print("✅ SensorAgent class defined")

✅ SensorAgent class defined


In [30]:
class ProcessingAgent:

    def __init__(self):
        self.agent_id = "ProcessingAgent"
        try:
            self.producer = KafkaProducer(
                bootstrap_servers=KAFKA_BROKER,
                value_serializer=serialize,
                max_block_ms=5000
            )
            self.consumer = KafkaConsumer(
                'sensor.data',
                bootstrap_servers=KAFKA_BROKER,
                value_deserializer=deserialize,
                auto_offset_reset='earliest',
                group_id='processing-group',
                consumer_timeout_ms=8000
            )
            print(f"🟢 {self.agent_id} initialized successfully")
        except Exception as e:
            print(f"❌ Error initializing {self.agent_id}: {e}")

    def process_and_publish(self):
        """Subscribe to sensor data, process it, and publish results"""
        print(f"👂 [{self.agent_id}] Listening to sensor data...")

        try:
            for message in self.consumer:
                sensor_data = message.value
                print(f"📥 [{self.agent_id}] Received: {sensor_data}")

                # Process the data
                processed = {
                    'sensor': sensor_data['sensor'],
                    'original_value': sensor_data['value'],
                    'processed_value': round(sensor_data['value'] * 1.1, 2),
                    'status': 'processed'
                }

                # Publish processed data
                self.producer.send('processed.data', processed)
                self.producer.flush()
                print(f"📤 [{self.agent_id}] Published processed: {processed['sensor']}")

                # Send command if temperature is high
                if sensor_data['sensor'] == 'temperature' and sensor_data['value'] > 20:
                    command = {'action': 'cooling_on', 'reason': 'high_temperature'}
                    self.producer.send('commands', command)
                    self.producer.flush()
                    print(f"📤 [{self.agent_id}] Published command: {command}")
        except:
            pass

    def run(self):
        """Execute agent"""
        self.process_and_publish()
        self.producer.close()
        self.consumer.close()
        print(f"✅ [{self.agent_id}] finished\n")

print("✅ ProcessingAgent class defined")

✅ ProcessingAgent class defined


In [31]:

class AnalyticsAgent:


    def __init__(self):
        self.agent_id = "AnalyticsAgent"
        self.data_count = 0
        try:
            self.producer = KafkaProducer(
                bootstrap_servers=KAFKA_BROKER,
                value_serializer=serialize,
                max_block_ms=5000
            )
            self.consumer = KafkaConsumer(
                'processed.data',
                'sensor.data',
                bootstrap_servers=KAFKA_BROKER,
                value_deserializer=deserialize,
                auto_offset_reset='earliest',
                group_id='analytics-group',
                consumer_timeout_ms=8000
            )
            print(f"🟡 {self.agent_id} initialized successfully")
        except Exception as e:
            print(f"❌ Error initializing {self.agent_id}: {e}")

    def analyze_and_publish(self):
        """Subscribe to multiple topics and publish analytics"""
        print(f"👂 [{self.agent_id}] Listening to multiple topics...")

        try:
            for message in self.consumer:
                data = message.value
                topic = message.topic
                self.data_count += 1

                print(f"📥 [{self.agent_id}] Received from {topic}: {data}")

                # Generate analytics report every 3 messages
                if self.data_count % 3 == 0:
                    report = {
                        'total_messages': self.data_count,
                        'timestamp': time.time(),
                        'status': 'healthy'
                    }
                    self.producer.send('analytics.report', report)
                    self.producer.flush()
                    print(f"📤 [{self.agent_id}] Published analytics report")
        except:
            pass

    def run(self):
        """Execute agent"""
        self.analyze_and_publish()
        self.producer.close()
        self.consumer.close()
        print(f"✅ [{self.agent_id}] finished\n")

print("✅ AnalyticsAgent class defined")

✅ AnalyticsAgent class defined


In [32]:

def run_all_agents():
    """Execute all 3 agents concurrently using threads"""

    print("\n" + "="*70)
    print("  KAFKA PUB/SUB MULTI-AGENT SYSTEM")
    print("  All 3 Agents: Publishers + Subscribers")
    print("="*70 + "\n")

    # Initialize all agents
    sensor = SensorAgent()
    processing = ProcessingAgent()
    analytics = AnalyticsAgent()

    # Define thread functions
    def run_processing():
        processing.run()

    def run_analytics():
        analytics.run()

    # Create threads
    processing_thread = threading.Thread(target=run_processing)
    analytics_thread = threading.Thread(target=run_analytics)

    # Start subscriber agents first
    print("🚀 Starting Processing and Analytics agents...\n")
    processing_thread.start()
    analytics_thread.start()
    time.sleep(2)

    # Start sensor agent
    print("🚀 Starting Sensor agent...\n")
    sensor_thread = threading.Thread(target=sensor.run)
    sensor_thread.start()

    # Wait for completion
    time.sleep(12)

    print("\n" + "="*70)
    print("  ✅ DEMONSTRATION COMPLETE")
    print("="*70)

# Execute the system
run_all_agents()


  KAFKA PUB/SUB MULTI-AGENT SYSTEM
  All 3 Agents: Publishers + Subscribers

🔵 SensorAgent initialized successfully
🟢 ProcessingAgent initialized successfully
🟡 AnalyticsAgent initialized successfully
🚀 Starting Processing and Analytics agents...

👂 [ProcessingAgent] Listening to sensor data...
👂 [AnalyticsAgent] Listening to multiple topics...


📥 [AnalyticsAgent] Received from sensor.data: {'sensor': 'temperature', 'value': 25.5}📥 [ProcessingAgent] Received: {'sensor': 'temperature', 'value': 25.5}

📥 [AnalyticsAgent] Received from sensor.data: {'sensor': 'humidity', 'value': 60.0}
📥 [AnalyticsAgent] Received from sensor.data: {'sensor': 'pressure', 'value': 1013.2}
📤 [ProcessingAgent] Published processed: temperature


📤 [AnalyticsAgent] Published analytics report
📥 [AnalyticsAgent] Received from processed.data: {'sensor': 'temperature', 'original_value': 25.5, 'processed_value': 28.05, 'status': 'processed'}
📤 [ProcessingAgent] Published command: {'action': 'cooling_on', 'reason': 'high_temperature'}
📥 [ProcessingAgent] Received: {'sensor': 'humidity', 'value': 60.0}
📤 [ProcessingAgent] Published processed: humidity
📥 [ProcessingAgent] Received: {'sensor': 'pressure', 'value': 1013.2}
📥 [AnalyticsAgent] Received from processed.data: {'sensor': 'humidity', 'original_value': 60.0, 'processed_value': 66.0, 'status': 'processed'}
📤 [ProcessingAgent] Published processed: pressure
📥 [AnalyticsAgent] Received from processed.data: {'sensor': 'pressure', 'original_value': 1013.2, 'processed_value': 1114.52, 'status': 'processed'}
📤 [AnalyticsAgent] Published analytics report
🚀 Starting Sensor agent...

📤 [SensorAgent] Published: {'sensor': 'temperature', 'value': 25.5}
📥 [ProcessingAgent] Received: {'sensor'

In [33]:

print("""
╔════════════════════════════════════════════════════════════════════╗
║          PUB/SUB ARCHITECTURE - 3 AGENTS DEMONSTRATION             ║
╚════════════════════════════════════════════════════════════════════╝

AGENT 1: SENSOR AGENT
────────────────────────────────────────────────────────────────────
  • Publishes to:   sensor.data (temperature, humidity, pressure)
  • Subscribes to:  commands (receives cooling/heating commands)
  • Purpose:        IoT sensor simulation

AGENT 2: PROCESSING AGENT
────────────────────────────────────────────────────────────────────
  • Subscribes to:  sensor.data
  • Publishes to:   processed.data (processed sensor values)
                    commands (control commands based on conditions)
  • Purpose:        Data processing and decision making

AGENT 3: ANALYTICS AGENT
────────────────────────────────────────────────────────────────────
  • Subscribes to:  processed.data, sensor.data (multiple topics)
  • Publishes to:   analytics.report (statistical summaries)
  • Purpose:        Data analysis and reporting

KAFKA TOPICS USED
────────────────────────────────────────────────────────────────────
  ✓ sensor.data      → Raw sensor measurements
  ✓ processed.data   → Processed sensor data
  ✓ commands         → Control commands
  ✓ analytics.report → Statistical reports

PUB/SUB PRINCIPLES DEMONSTRATED
────────────────────────────────────────────────────────────────────
  ✓ Asynchronous communication - agents don't wait for responses
  ✓ Loose coupling - publishers don't know subscribers
  ✓ Scalability - easy to add more agents
  ✓ Central broker - Kafka manages all message routing
  ✓ Topic-based routing - messages organized by topics
  ✓ Multiple subscribers - multiple agents can read same topic
""")


╔════════════════════════════════════════════════════════════════════╗
║          PUB/SUB ARCHITECTURE - 3 AGENTS DEMONSTRATION             ║
╚════════════════════════════════════════════════════════════════════╝

AGENT 1: SENSOR AGENT
────────────────────────────────────────────────────────────────────
  • Publishes to:   sensor.data (temperature, humidity, pressure)
  • Subscribes to:  commands (receives cooling/heating commands)
  • Purpose:        IoT sensor simulation

AGENT 2: PROCESSING AGENT
────────────────────────────────────────────────────────────────────
  • Subscribes to:  sensor.data
  • Publishes to:   processed.data (processed sensor values)
                    commands (control commands based on conditions)
  • Purpose:        Data processing and decision making

AGENT 3: ANALYTICS AGENT
────────────────────────────────────────────────────────────────────
  • Subscribes to:  processed.data, sensor.data (multiple topics)
  • Publishes to:   analytics.report (statistic

In [34]:

print("🧹 Cleaning up Kafka processes...")

try:
    kafka_server.terminate()
    zookeeper.terminate()
    time.sleep(2)
    print("✅ Kafka and Zookeeper stopped successfully")
except:
    print("ℹ️  Processes already stopped")

print("\n✨ Cleanup complete!")

🧹 Cleaning up Kafka processes...
✅ Kafka and Zookeeper stopped successfully

✨ Cleanup complete!
